In [1]:
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "gmail-triage-t24-qa"
NOTEBOOK_PATH = "notebooks/qa/47-gmail-triage-t24-qa.ipynb"
TOWER = 24
TOWER_NAME = "Tower of Gmail Inbox Triage"
MIN_SCORE = 70
BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"
results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

Tower 24: Tower of Gmail Inbox Triage


In [2]:
# Cell 1: Gmail recipe files exist and have required structure
print("=== Gmail Recipe Files ===")

# P1: Gmail read-inbox recipe exists
recipe_dir = BROWSER_ROOT / "data" / "default" / "recipes" / "gmail"
recipe_files = list(recipe_dir.glob("*.json")) if recipe_dir.exists() else []
record("T24-P01", "Gmail recipe directory exists with JSON files",
       recipe_dir.exists() and len(recipe_files) >= 1,
       f"Found {len(recipe_files)} recipe files in {recipe_dir}")

# P2: gmail-read-inbox recipe has oauth3_scopes
read_inbox = recipe_dir / "gmail-read-inbox.json"
content = read_file(read_inbox)
if content:
    data = json.loads(content)
    has_scopes = "oauth3_scopes" in data and len(data["oauth3_scopes"]) > 0
    record("T24-P02", "gmail-read-inbox recipe defines oauth3_scopes",
           has_scopes,
           f"scopes={data.get('oauth3_scopes', [])}")
else:
    record("T24-P02", "gmail-read-inbox recipe defines oauth3_scopes", False, "File not found")

# P3: gmail-read-inbox has steps array
if content:
    data = json.loads(content)
    has_steps = "steps" in data and len(data["steps"]) >= 2
    record("T24-P03", "gmail-read-inbox recipe has multiple steps",
           has_steps,
           f"steps_count={len(data.get('steps', []))}")
else:
    record("T24-P03", "gmail-read-inbox recipe has multiple steps", False, "File not found")

# P4: Multiple Gmail recipes exist (compose, search, reply, archive, label)
expected_gmail_recipes = ["gmail-compose-email.json", "gmail-search-emails.json",
                          "gmail-reply-email.json", "gmail-archive-emails.json",
                          "gmail-label-emails.json"]
found_recipes = [f for f in expected_gmail_recipes if (recipe_dir / f).exists()]
record("T24-P04", "Multiple Gmail recipes exist (compose, search, reply, archive, label)",
       len(found_recipes) >= 4,
       f"Found {len(found_recipes)}/{len(expected_gmail_recipes)}: {found_recipes}")

# P5: gmail-send-email legacy recipe exists at top-level
legacy_send = BROWSER_ROOT / "data" / "default" / "recipes" / "gmail-send-email.recipe.json"
record("T24-P05", "Legacy gmail-send-email recipe exists",
       legacy_send.exists(),
       str(legacy_send))

=== Gmail Recipe Files ===
PASS: Gmail recipe directory exists with JSON files -- Found 6 recipe files in /home/phuc/projects/solace-browser/data/default/recipes/gmail
PASS: gmail-read-inbox recipe defines oauth3_scopes -- scopes=['gmail.read.inbox']
PASS: gmail-read-inbox recipe has multiple steps -- steps_count=6
PASS: Multiple Gmail recipes exist (compose, search, reply, archive, label) -- Found 5/5: ['gmail-compose-email.json', 'gmail-search-emails.json', 'gmail-reply-email.json', 'gmail-archive-emails.json', 'gmail-label-emails.json']
PASS: Legacy gmail-send-email recipe exists -- /home/phuc/projects/solace-browser/data/default/recipes/gmail-send-email.recipe.json


In [3]:
# Cell 2: OAuth3 scopes for Gmail are registered
print("=== Gmail OAuth3 Scopes ===")

scopes_py = read_file(SRC / "oauth3" / "scopes.py")

# P6: gmail.read.inbox scope exists
record("T24-P06", "gmail.read.inbox scope registered in scopes.py",
       '"gmail.read.inbox"' in scopes_py,
       "Triple-segment scope for inbox read")

# P7: gmail.send.email scope exists (high risk)
record("T24-P07", "gmail.send.email scope registered as high risk",
       '"gmail.send.email"' in scopes_py and 'risk_level": "high"' in scopes_py.split('gmail.send.email')[1][:200] if 'gmail.send.email' in scopes_py else False,
       "High risk scope for sending emails")

# P8: gmail scopes include read, send, delete, label, draft, search
gmail_scopes = ["gmail.read.inbox", "gmail.read.labels", "gmail.send.email",
                "gmail.delete.email", "gmail.label.apply", "gmail.draft.create",
                "gmail.search.messages"]
found_scopes = [s for s in gmail_scopes if f'"{s}"' in scopes_py]
record("T24-P08", "All 7 Gmail scopes registered (read/send/delete/label/draft/search)",
       len(found_scopes) == 7,
       f"Found {len(found_scopes)}/7: {found_scopes}")

# P9: gmail scopes are in SCOPE_REGISTRY (triple-segment format)
record("T24-P09", "Gmail scopes follow triple-segment format (platform.action.resource)",
       all(len(s.split('.')) == 3 for s in found_scopes),
       "All found scopes have 3 segments")

=== Gmail OAuth3 Scopes ===
PASS: gmail.read.inbox scope registered in scopes.py -- Triple-segment scope for inbox read
PASS: gmail.send.email scope registered as high risk -- High risk scope for sending emails
PASS: All 7 Gmail scopes registered (read/send/delete/label/draft/search) -- Found 7/7: ['gmail.read.inbox', 'gmail.read.labels', 'gmail.send.email', 'gmail.delete.email', 'gmail.label.apply', 'gmail.draft.create', 'gmail.search.messages']
PASS: Gmail scopes follow triple-segment format (platform.action.resource) -- All found scopes have 3 segments


In [4]:
# Cell 3: PrimeWiki data for Gmail exists
print("=== Gmail PrimeWiki Data ===")

primewiki_gmail = BROWSER_ROOT / "data" / "default" / "primewiki" / "gmail"

# P10: Gmail primewiki directory exists
record("T24-P10", "Gmail PrimeWiki directory exists",
       primewiki_gmail.exists() and primewiki_gmail.is_dir(),
       str(primewiki_gmail))

# P11: Gmail selectors.json exists
selectors = primewiki_gmail / "selectors.json"
record("T24-P11", "Gmail selectors.json exists in PrimeWiki",
       selectors.exists(),
       str(selectors))

# P12: Gmail actions.json exists
actions = primewiki_gmail / "actions.json"
record("T24-P12", "Gmail actions.json exists in PrimeWiki",
       actions.exists(),
       str(actions))

# P13: Gmail urls.json exists
urls = primewiki_gmail / "urls.json"
record("T24-P13", "Gmail urls.json exists in PrimeWiki",
       urls.exists(),
       str(urls))

# P14: Gmail primewiki has page flow mermaid
mermaid_file = primewiki_gmail / "gmail-page-flow.prime-mermaid.md"
record("T24-P14", "Gmail page flow Mermaid diagram exists",
       mermaid_file.exists(),
       str(mermaid_file))

=== Gmail PrimeWiki Data ===
PASS: Gmail PrimeWiki directory exists -- /home/phuc/projects/solace-browser/data/default/primewiki/gmail
PASS: Gmail selectors.json exists in PrimeWiki -- /home/phuc/projects/solace-browser/data/default/primewiki/gmail/selectors.json
PASS: Gmail actions.json exists in PrimeWiki -- /home/phuc/projects/solace-browser/data/default/primewiki/gmail/actions.json
PASS: Gmail urls.json exists in PrimeWiki -- /home/phuc/projects/solace-browser/data/default/primewiki/gmail/urls.json
PASS: Gmail page flow Mermaid diagram exists -- /home/phuc/projects/solace-browser/data/default/primewiki/gmail/gmail-page-flow.prime-mermaid.md


In [5]:
# Cell 4: Recipe engine and evidence infrastructure
print("=== Recipe Engine + Evidence ===")

# P15: Recipe executor module exists
recipe_executor = SRC / "recipes" / "recipe_executor.py"
record("T24-P15", "Recipe executor module exists",
       recipe_executor.exists(),
       str(recipe_executor))

# P16: Recipe parser module exists
recipe_parser = SRC / "recipes" / "recipe_parser.py"
record("T24-P16", "Recipe parser module exists",
       recipe_parser.exists(),
       str(recipe_parser))

# P17: Evidence chain module exists with SHA-256
evidence_chain = SRC / "audit" / "chain.py"
chain_content = read_file(evidence_chain)
record("T24-P17", "Evidence chain module with SHA-256 hashing",
       "sha256" in chain_content.lower() and "hashlib" in chain_content,
       "SHA-256 hash chain for tamper-evident audit")

# P18: OAuth3 evidence chain module exists
oauth3_evidence = SRC / "oauth3" / "evidence.py"
evidence_content = read_file(oauth3_evidence)
record("T24-P18", "OAuth3 evidence chain with hash linking",
       "prev_hash" in evidence_content and "GENESIS_HASH" in evidence_content,
       "Hash-chained JSONL audit log for OAuth3 events")

# P19: Evidence summary formatter exists (plain-English output)
summary_fmt = SRC / "evidence" / "summary_formatter.py"
fmt_content = read_file(summary_fmt)
record("T24-P19", "Evidence summary formatter for plain-English output",
       "format_action_summary" in fmt_content and "format_step_timing" in fmt_content,
       "Converts raw actions to human-readable summaries")

# P20: App store page references /api/apps route
app_store_html = read_file(WEB / "app-store.html")
record("T24-P20", "App store page has search input for discovering Gmail app",
       'id="app-search"' in app_store_html and 'type="search"' in app_store_html,
       "Search input allows users to find Gmail app")

=== Recipe Engine + Evidence ===
PASS: Recipe executor module exists -- /home/phuc/projects/solace-browser/src/recipes/recipe_executor.py
PASS: Recipe parser module exists -- /home/phuc/projects/solace-browser/src/recipes/recipe_parser.py
PASS: Evidence chain module with SHA-256 hashing -- SHA-256 hash chain for tamper-evident audit
PASS: OAuth3 evidence chain with hash linking -- Hash-chained JSONL audit log for OAuth3 events
PASS: Evidence summary formatter for plain-English output -- Converts raw actions to human-readable summaries
PASS: App store page has search input for discovering Gmail app -- Search input allows users to find Gmail app


In [6]:
# Evidence summary cell
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"
summary = {"surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME, "timestamp": datetime.now().isoformat(), "total_probes": total, "passed": passed, "failed": failed, "score": score, "grade": grade, "finding_rate": finding_rate, "converged": finding_rate < 5, "findings": [r for r in results if r["status"] == "FAIL"], "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]}
print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))

TOWER 24: Tower of Gmail Inbox Triage
Probes: 20 | Passed: 20 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: ed0fd859900db176

{
  "surface": "gmail-triage-t24-qa",
  "tower": 24,
  "tower_name": "Tower of Gmail Inbox Triage",
  "timestamp": "2026-03-06T11:29:07.404652",
  "total_probes": 20,
  "passed": 20,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "ed0fd859900db176"
}
